# Task 1 — inconsistent values and duplicates

The first cells display the original CSV without changing it. The later cells clean a copy, count both kinds of duplicates, and save the result.

In [19]:
import pandas as pd

# Read every field as text so the first inspection shows the original CSV values.
raw = pd.read_csv("club_signups.csv", dtype="string")

In [20]:
def show_overview(data):
    """Print the size, data types, missing-value counts, and every row."""
    print("Shape (rows, columns):", data.shape)
    print("\nData types:")
    print(data.dtypes.to_string())
    print("\nMissing values:")
    print(data.isna().sum().to_string())
    print("\nAll original rows:")
    print(data.to_string(index=False))


show_overview(raw)

Shape (rows, columns): (39, 9)

Data types:
signup_id       string[python]
student_id      string[python]
full_name       string[python]
email           string[python]
faculty         string[python]
city            string[python]
club            string[python]
signed_up_at    string[python]
fee_paid        string[python]

Missing values:
signup_id       0
student_id      0
full_name       0
email           0
faculty         0
city            0
club            0
signed_up_at    0
fee_paid        0

All original rows:
signup_id student_id          full_name                             email                                faculty       city        club     signed_up_at fee_paid
     S015    61-8942          Hana Zaki      hana.zaki@student.guc.edu.eg                               Pharmacy       Giza      Soccer 2026-09-15 13:39        N
     S010    61-3846         AHMED ZAKI     ahmed.zaki@student.guc.edu.eg                             Management alexandria       Chess 15/09/2026 03:12  

In [21]:
def show_value_counts(data, columns):
    """Print every distinct value and its frequency for selected columns."""
    for column in columns:
        if column not in data.columns:
            raise KeyError(f"Unknown column: {column}")
        print(f"\n{column} — value counts:")
        print(data[column].value_counts(dropna=False).to_string())


# Categories and person details; no values are changed here.
show_value_counts(raw, ["faculty", "city", "club", "fee_paid", "full_name", "email"])


faculty — value counts:
faculty
Pharmacy                                  7
EMS                                       7
IET                                       5
Management                                4
Information Engineering and Technology    4
MET                                       3
iet                                       3
Engineering and Materials Science         1
Pharma                                    1
met                                       1
Media Engineering and Technology          1
pharmacy                                  1
ems                                       1

city — value counts:
city
Cairo         13
Alexandria    11
Giza           4
Alex           4
cairo          2
giza           2
alexandria     1
CAIRO          1
El Giza        1

club — value counts:
club
Robotics       8
Music          5
Chess          4
football       4
Debating       3
Chess Club     3
Football       3
Soccer         2
music          2
chess          2
debate club    1
M

In [22]:
# IDs and submitted timestamps can also help you spot repeated records.
show_value_counts(raw, ["signup_id", "student_id", "signed_up_at"])


signup_id — value counts:
signup_id
S015    2
S022    2
S023    2
S028    1
S017    1
S031    1
S014    1
S011    1
S018    1
S030    1
S016    1
S029    1
S034    1
S009    1
S021    1
S013    1
S032    1
S019    1
S024    1
S010    1
S020    1
S008    1
S035    1
S004    1
S012    1
S007    1
S005    1
S033    1
S001    1
S003    1
S006    1
S026    1
S025    1
S002    1
S027    1
S036    1

student_id — value counts:
student_id
61-7344    6
55-9245    5
61-4844    4
61-8942    3
58-1125    3
61-8224    3
61-2495    2
61-4733    2
55-2992    2
58-8701    2
58-1089    2
55-2639    2
61-3846    1
58-9118    1
55-2971    1

signed_up_at — value counts:
signed_up_at
2026-09-15 13:39    2
2026-09-16 01:39    2
2026-09-16 09:29    2
2026-09-17 06:54    1
15/09/2026 18:01    1
2026-09-17 23:29    1
2026-09-15 11:36    1
15/09/2026 03:28    1
15/09/2026 18:40    1
2026-09-17 17:14    1
15/09/2026 17:20    1
17/09/2026 12:27    1
16/09/2026 02:01    1
2026-09-15 01:05    1
16/09/2026 01:15  

## What we found

Faculty, city, club, fee and person details have inconsistent spellings or formats. The timestamps also use two formats. Repeated student IDs are expected when a student joins different clubs, so the deduplication key will be `student_id` together with the cleaned `club`.

## Part A — standardize categories

The same helper first strips spaces and ignores case, then applies an explicit dictionary. It rejects unknown values instead of silently leaving them unchanged.

In [23]:
def standardize_category(series, mapping, allowed):
    """Normalize text, map known spellings, and reject unexpected values."""
    normalized = series.str.strip().str.casefold()
    unknown = sorted(set(normalized.dropna()) - set(mapping))
    if unknown:
        raise ValueError(f"Unexpected {series.name} values: {unknown}")
    result = normalized.map(mapping)
    assert result.notna().all(), f"Missing or unmapped values in {series.name}"
    assert set(result) <= allowed, f"Non-canonical values in {series.name}"
    return result


category_rules = {
    "faculty": (
        {
            "met": "MET", "media engineering and technology": "MET",
            "iet": "IET", "information engineering and technology": "IET",
            "ems": "EMS", "engineering and materials science": "EMS",
            "pharmacy": "Pharmacy", "pharma": "Pharmacy",
            "management": "Management",
        },
        {"MET", "IET", "EMS", "Pharmacy", "Management"},
    ),
    "city": (
        {
            "cairo": "Cairo", "giza": "Giza", "el giza": "Giza",
            "alexandria": "Alexandria", "alex": "Alexandria",
        },
        {"Cairo", "Giza", "Alexandria"},
    ),
    "club": (
        {
            "robotics": "Robotics",
            "debate": "Debate", "debating": "Debate", "debate club": "Debate",
            "music": "Music", "football": "Football", "soccer": "Football",
            "chess": "Chess", "chess club": "Chess",
        },
        {"Robotics", "Debate", "Music", "Football", "Chess"},
    ),
    "fee_paid": (
        {"yes": True, "y": True, "true": True, "1": True,
         "no": False, "n": False, "false": False, "0": False},
        {True, False},
    ),
}

clean = raw.copy()
for column, (mapping, allowed) in category_rules.items():
    clean[column] = standardize_category(clean[column], mapping, allowed)
clean["fee_paid"] = clean["fee_paid"].astype(bool)
assert pd.api.types.is_bool_dtype(clean["fee_paid"])
show_value_counts(clean, ["faculty", "city", "club", "fee_paid"])


faculty — value counts:
faculty
IET           12
Pharmacy       9
EMS            9
MET            5
Management     4

city — value counts:
city
Alexandria    16
Cairo         16
Giza           7

club — value counts:
club
Football    9
Chess       9
Music       8
Robotics    8
Debate      5

fee_paid — value counts:
fee_paid
True     27
False    12


## Part A — names, emails, and timestamps

Names use one space between words and title case. Emails are trimmed and lower-case. Slash-separated dates are day/month/year; values such as `18/09/2026` make the day/month order unambiguous.

In [24]:
clean["full_name"] = clean["full_name"].str.split().str.join(" ").str.title()
clean["email"] = clean["email"].str.strip().str.lower()

assert clean["full_name"].eq(clean["full_name"].str.strip()).all()
assert not clean["full_name"].str.contains(r"\s{2,}").any()
assert clean["email"].eq(clean["email"].str.strip().str.lower()).all()

name_changes = pd.DataFrame({"before": raw["full_name"], "after": clean["full_name"]})
email_changes = pd.DataFrame({"before": raw["email"], "after": clean["email"]})
print("Names changed:")
print(name_changes.loc[name_changes["before"] != name_changes["after"]].to_string(index=False))
print("\nEmails changed:")
print(email_changes.loc[email_changes["before"] != email_changes["after"]].to_string(index=False))

Names changed:
            before          after
        AHMED ZAKI     Ahmed Zaki
    Mohamed  Adel    Mohamed Adel
      MOHAMED ADEL   Mohamed Adel
       ALI IBRAHIM    Ali Ibrahim
        ahmed adel     Ahmed Adel
         OMAR ADEL      Omar Adel
       ali ibrahim    Ali Ibrahim
       ali ibrahim    Ali Ibrahim
        AHMED ADEL     Ahmed Adel
       Omar  Adel       Omar Adel
      mohamed adel   Mohamed Adel
   Laila  Mostafa   Laila Mostafa
  Farida  Ibrahim  Farida Ibrahim
     laila mostafa  Laila Mostafa

Emails changed:
                           before                            after
    AHMED.ADEL@STUDENT.GUC.EDU.EG    ahmed.adel@student.guc.edu.eg
  MOHAMED.ADEL@STUDENT.GUC.EDU.EG  mohamed.adel@student.guc.edu.eg
 mohamed.adel@student.guc.edu.eg   mohamed.adel@student.guc.edu.eg
 LAILA.MOSTAFA@STUDENT.GUC.EDU.EG laila.mostafa@student.guc.edu.eg
 LAILA.MOSTAFA@STUDENT.GUC.EDU.EG laila.mostafa@student.guc.edu.eg


In [25]:
timestamps = clean["signed_up_at"].str.strip()
iso = timestamps.str.fullmatch(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}")
day_first = timestamps.str.fullmatch(r"\d{2}/\d{2}/\d{4} \d{2}:\d{2}")
assert (iso | day_first).all() and not (iso & day_first).any()

parsed = pd.Series(pd.NaT, index=clean.index, dtype="datetime64[ns]")
parsed.loc[iso] = pd.to_datetime(timestamps.loc[iso], format="%Y-%m-%d %H:%M", errors="raise")
parsed.loc[day_first] = pd.to_datetime(timestamps.loc[day_first], format="%d/%m/%Y %H:%M", errors="raise")
clean["signed_up_at"] = parsed
assert clean["signed_up_at"].notna().all()
assert pd.api.types.is_datetime64_any_dtype(clean["signed_up_at"])
print(f"ISO timestamps: {int(iso.sum())}; day/month/year timestamps: {int(day_first.sum())}")
print("Parsed date range:", clean["signed_up_at"].min(), "to", clean["signed_up_at"].max())

ISO timestamps: 25; day/month/year timestamps: 14
Parsed date range: 2026-09-13 14:36:00 to 2026-09-18 23:36:00


## Part B — exact copies and signup history

First remove rows that are identical in every column. The remaining repeated student–club entries are separate submissions: they show the history of a signup, often with an updated fee status. The assignment asks for one final row per `student_id` and cleaned `club`, so we display that history and keep the latest submission in the output. A repeated `student_id` in a different club stays.

In [26]:
exact_removed = int(clean.duplicated().sum())
unique_submissions = clean.drop_duplicates().copy()
print(f"Exact duplicates removed: {exact_removed}")
print(f"Rows after exact duplicates: {len(unique_submissions)}")

key = ["student_id", "club"]
repeated_signups = unique_submissions.loc[
    unique_submissions.duplicated(key, keep=False),
    ["signup_id", "student_id", "club", "signed_up_at", "fee_paid"],
].sort_values(key + ["signed_up_at"])
print("\nRepeated student–club submissions before keeping the latest:")
print(repeated_signups.to_string(index=False))

repeat_removed = int(unique_submissions.duplicated(key).sum())
cleaned = (
    unique_submissions.sort_values("signed_up_at", kind="stable")
    .drop_duplicates(key, keep="last")
    .sort_values(key)
    .reset_index(drop=True)
)
assert not cleaned.duplicated(key).any()
assert len(cleaned) == len(raw) - exact_removed - repeat_removed
print(f"\nEarlier student–club submissions removed: {repeat_removed}")
print(f"Rows remaining: {len(cleaned)}")

Exact duplicates removed: 3
Rows after exact duplicates: 36

Repeated student–club submissions before keeping the latest:
signup_id student_id     club        signed_up_at  fee_paid
     S020    55-9245    Music 2026-09-15 23:07:00     False
     S033    55-9245    Music 2026-09-16 09:07:00      True
     S021    61-4844   Debate 2026-09-16 01:15:00     False
     S035    61-4844   Debate 2026-09-17 11:15:00      True
     S017    61-7344   Debate 2026-09-15 18:01:00     False
     S034    61-7344   Debate 2026-09-16 02:01:00      True
     S032    61-7344 Football 2026-09-17 23:36:00     False
     S036    61-7344 Football 2026-09-18 23:36:00      True

Earlier student–club submissions removed: 4
Rows remaining: 32


In [27]:
# Compare with deduplicating before club spellings are standardized.
raw_unique = raw.drop_duplicates()
early_repeat_count = int(raw_unique.duplicated(["student_id", "club"]).sum())
print(f"Repeated student–club rows with raw club spellings: {early_repeat_count}")
print(f"Repeated student–club rows with canonical club names: {repeat_removed}")
print(f"Missed if club names are not standardized first: {repeat_removed - early_repeat_count}")

Repeated student–club rows with raw club spellings: 2
Repeated student–club rows with canonical club names: 4
Missed if club names are not standardized first: 2


## Save and check the final table

In [28]:
output_path = "club_signups_clean.csv"
cleaned.to_csv(output_path, index=False)
print(f"Saved {len(cleaned)} rows to {output_path}")
print("\nFinal data types:")
print(cleaned.dtypes.to_string())
print("\nFinal table:")
print(cleaned.to_string(index=False))

Saved 32 rows to club_signups_clean.csv

Final data types:
signup_id       string[python]
student_id      string[python]
full_name               object
email           string[python]
faculty                 object
city                    object
club                    object
signed_up_at    datetime64[ns]
fee_paid                  bool

Final table:
signup_id student_id      full_name                             email    faculty       city     club        signed_up_at  fee_paid
     S014    55-2639  Youssef Fathy  youssef.fathy@student.guc.edu.eg Management      Cairo    Chess 2026-09-15 11:36:00      True
     S016    55-2639  Youssef Fathy  youssef.fathy@student.guc.edu.eg Management      Cairo Robotics 2026-09-15 17:20:00      True
     S030    55-2971  Salma Ibrahim  salma.ibrahim@student.guc.edu.eg Management Alexandria    Chess 2026-09-17 17:14:00     False
     S002    55-2992   Mohamed Adel   mohamed.adel@student.guc.edu.eg        IET Alexandria   Debate 2026-09-13 15:38:00    